# Teaching LLMs to Play Games with GRPO + OpenEnv

## 🎯 The Vision

Imagine training language models to:
- 🎰 **Play BlackJack** with near-optimal strategy
- ♟️ **Master Chess** through self-play
- 📈 **Trade stocks** in realistic market simulations  
- 🎮 **Beat Atari games** like human experts
- 💻 **Debug code** in interactive programming environments

---

### The Problem

Connecting LLMs to game environments has been painful:
- ❌ Installing game engines locally (OpenSpiel C++, Atari emulators, trading sims)
- ❌ Handling different APIs for each environment
- ❌ Managing dependencies, versions, OS compatibility
- ❌ No isolation → training crashes can corrupt your system

### The Solution: OpenEnv + Forge

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; margin: 20px 0;'>
    <h3 style='margin-top: 0;'>🚀 OpenEnv = Universal Game Environment Connector</h3>
    <ul style='font-size: 16px;'>
        <li><b>70+ games</b> (OpenSpiel, Atari, FinRL, etc.)</li>
        <li><b>Clean Gymnasium API:</b> <code>reset()</code>, <code>step(action)</code>, <code>state()</code></li>
        <li><b>Docker-isolated</b> → reproducible, secure, scalable</li>
        <li><b>HTTP-based</b> → language-agnostic (Python, Rust, whatever)</li>
    </ul>
</div>

<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 20px; border-radius: 10px; color: white; margin: 20px 0;'>
    <h3 style='margin-top: 0;'>⚡ Forge + GRPO = Production RL Training</h3>
    <ul style='font-size: 16px;'>
        <li><b>GRPO:</b> Group Relative Policy Optimization for stable LLM training</li>
        <li><b>Distributed infrastructure:</b> vLLM, TorchTitan, replay buffers</li>
        <li><b>Built by Meta</b> for training large models at scale</li>
    </ul>
</div>

---

## What You'll Build

In this notebook, you'll train a **Qwen 1.5B model** to play BlackJack using **production GRPO code**.

**The Journey:**
1. 🔌 **Connect** to BlackJack via OpenEnv
2. 🏗️ **Initialize** Forge infrastructure (vLLM, Trainer, ReplayBuffer)
3. 🔥 **Train** with actual GRPO implementation
4. 📊 **Monitor** training metrics in real-time
5. 🎬 **Evaluate** the trained policy

**This is the REAL code.** Same as `apps/grpo/blackjack_main_fixed.py`. 🚀

---

### 📚 Further Reading
- 📦 [OpenEnv GitHub](https://github.com/meta-pytorch/OpenEnv)
- 📄 [GRPO Paper (arXiv:2402.03300)](https://arxiv.org/abs/2402.03300)  
- 🔧 [Forge Repository](https://github.com/meta-pytorch/forge)

## 🏗️ Architecture: How the Pieces Connect

<div style='background: #f8f9fa; padding: 20px; border-radius: 10px; border-left: 5px solid #667eea;'>
<pre style='font-size: 14px; line-height: 1.6;'>
┌─────────────────────────────────────────────────┐
│   <b>Your Training Loop</b> (This Notebook)         │
│                                                 │
│  ┌──────────────────────────────────────────┐  │
│  │  <b>Continuous Rollouts</b>                  │  │
│  │  • Play games with policy            │  │
│  │  • Collect episodes                  │  │
│  │  • Compute advantages                │  │
│  │  • Add to replay buffer              │  │
│  └──────────────────────────────────────────┘  │
│                                                 │
│  ┌──────────────────────────────────────────┐  │
│  │  <b>Continuous Training</b>                  │  │
│  │  • Sample batch from buffer          │  │
│  │  • Compute GRPO loss                 │  │
│  │  • Update policy weights             │  │
│  │  • Push to all replicas              │  │
│  └──────────────────────────────────────────┘  │
│                                                 │
└────────┬──────────────────────┬─────────────────┘
         │                      │
         │ HTTP                 │ Distributed RPC
         │                      │
┌────────▼──────────┐  ┌────────▼──────────────┐
│  <b>OpenEnv Server</b>  │  │  <b>Forge Services</b>      │
│  (BlackJack)      │  │  • Generator (vLLM)   │
│                   │  │  • RLTrainer          │
│                   │  │  • ReplayBuffer       │
│                   │  │  • ReferenceModel     │
└───────────────────┘  └───────────────────────┘
</pre>
</div>

## 🚀 Step 1: Start OpenEnv Server

First, start the BlackJack environment server.

<div style='background: #fff3cd; padding: 15px; border-radius: 8px; border-left: 5px solid #ffc107; margin: 20px 0;'>
    <b>⚠️ Note:</b> Make sure you have OpenEnv installed and the path is correct.
    You can start the server manually in a separate terminal:
    <pre style='margin-top: 10px; background: white; padding: 10px; border-radius: 5px;'>
export PYTHONPATH="/Users/sanyambhutani/OpenEnv/OpenEnv/src:${PYTHONPATH}"
OPENSPIEL_GAME=blackjack python -m envs.openspiel_env.server.app</pre>
</div>

In [ ]:
import subprocess
import time
import requests
import os
import sys

def start_openenv_server(port=8004):
    """Start OpenEnv BlackJack server."""
    env_vars = os.environ.copy()
    
    openenv_path = "/Users/sanyambhutani/OpenEnv/OpenEnv/src"
    env_vars["PYTHONPATH"] = f"{openenv_path}:{env_vars.get('PYTHONPATH', '')}"
    env_vars["OPENSPIEL_GAME"] = "blackjack"
    
    print(f"🚀 Starting OpenEnv server on port {port}...")
    
    process = subprocess.Popen(
        [sys.executable, "-m", "envs.openspiel_env.server.app"],
        env=env_vars,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    
    # Wait for server
    server_url = f"http://localhost:{port}"
    for i in range(30):
        try:
            response = requests.get(f"{server_url}/health", timeout=1)
            if response.status_code == 200:
                print(f"✅ Server ready at {server_url}")
                return process
        except requests.exceptions.RequestException:
            time.sleep(1)
            if i % 5 == 0 and i > 0:
                print(f"   ⏳ Waiting... ({i}s)")
    
    process.kill()
    raise TimeoutError("Server failed to start")

# Start server (comment this out if running manually)
# server_process = start_openenv_server(port=8004)
print("✅ Assuming OpenEnv server is running at http://localhost:8004")

## 📦 Step 2: Imports and Configuration

Import the production GRPO utilities and Forge components.

In [ ]:
import asyncio
import uuid
from pathlib import Path

import torch
import torchstore as ts
from omegaconf import OmegaConf

# Import production GRPO utilities
from apps.grpo.grpo_utils import (
    Episode,
    Group,
    collate,
    simple_grpo_loss,
    BlackJackReward,
    ComputeAdvantages,
    BlackJackEnvActor,
    setup_game_logger,
    drop_weights,
    play_blackjack_game,
)

# Import Forge infrastructure
from forge.actors.generator import Generator
from forge.actors.reference_model import ReferenceModel
from forge.actors.replay_buffer import ReplayBuffer
from forge.actors.trainer import RLTrainer
from forge.controller.provisioner import init_provisioner, shutdown
from forge.observability.metric_actors import get_or_create_metric_logger
from forge.observability.metrics import record_metric, Reduce
from forge.observability.perf_tracker import Tracer
from forge.types import LauncherConfig, ProvisionerConfig

print("✅ Imports successful")

## ⚙️ Step 3: Load Configuration

Load the production BlackJack GRPO configuration.

In [ ]:
# Load config from YAML
config_path = Path("apps/grpo/blackjack.yaml")
cfg = OmegaConf.load(config_path)

# Override some settings for notebook training
cfg.trainer.training.steps = 100  # Shorter training for demo
cfg.group_size = 4  # Play 4 games per rollout
cfg.rollout_threads = 1  # Single rollout thread

print("📋 Configuration:")
print(f"  Model: {cfg.blackjack_env.model}")
print(f"  Training steps: {cfg.trainer.training.steps}")
print(f"  Group size: {cfg.group_size}")
print(f"  Batch size: {cfg.training.local_batch_size}")
print(f"  OpenEnv URL: {cfg.blackjack_env.server_url}")

## 📚 Understanding GRPO

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 10px; color: white; margin: 20px 0;'>
    <h3 style='margin-top: 0;'>🧠 GRPO: Group Relative Policy Optimization</h3>
    <p style='font-size: 16px; line-height: 1.6;'>
        <b>Key Idea:</b> Instead of comparing an episode to a fixed baseline, compare it to a <b>group of episodes</b> collected at the same time.
    </p>
    <p style='font-size: 16px; line-height: 1.6;'>
        <b>Why?</b> More stable training for LLMs. Group statistics provide better normalization than a slowly-updating baseline.
    </p>
</div>

### Core Components

Let's look at the key data structures and loss function from our utils.

In [ ]:
# Show the GRPO loss function (from grpo_utils.py)
from IPython.display import Code

loss_code = '''
def simple_grpo_loss(
    logits: torch.Tensor,
    response: torch.Tensor,
    ref_logprobs: torch.Tensor,
    advantages: torch.Tensor,
    padding_mask: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    """GRPO loss with KL penalty."""
    
    # Compute log probabilities
    logprobs = compute_logprobs(logits, response)
    
    # KL divergence: KL(ref || policy) in closed form
    kl = torch.exp(ref_logprobs - logprobs) - (ref_logprobs - logprobs) - 1
    
    # Policy gradient with importance weight
    per_token_policy_loss = torch.exp(logprobs - logprobs.detach()) * advantages
    
    # Combined: maximize policy improvement, minimize KL
    per_token_loss = -(per_token_policy_loss - beta * kl)
    
    # Average over valid tokens
    return ((per_token_loss * padding_mask).sum(dim=1) / 
            padding_mask.sum(dim=1).clamp(min=1.0)).mean()
'''

display(Code(loss_code, language='python'))

print("\n📊 Key Points:")
print("  1. Policy gradient scaled by advantages (group-relative)")
print("  2. KL penalty keeps policy close to reference")
print("  3. Per-token computation with proper masking")

## 🏗️ Step 4: Initialize Forge Infrastructure

Now let's spin up all the Forge services!

In [ ]:
async def initialize_services(cfg):
    """Initialize all Forge services and actors."""
    
    print("🏗️ Initializing Forge infrastructure...\n")
    
    # Initialize provisioner
    if cfg.get("provisioner", None) is not None:
        provisioner = await init_provisioner(
            ProvisionerConfig(launcher_config=LauncherConfig(**cfg.provisioner))
        )
    else:
        provisioner = await init_provisioner()
    print("  ✅ Provisioner")
    
    # Initialize metric logging
    metric_logging_cfg = cfg.get("metric_logging", {"console": {"log_per_rank": False}})
    mlogger = await get_or_create_metric_logger()
    await mlogger.init_backends.call_one(metric_logging_cfg)
    print("  ✅ Metric Logger")
    
    # Initialize all services in parallel
    print("\n  🚀 Initializing services (this may take a minute)...")
    (
        blackjack_env,
        policy,
        trainer,
        replay_buffer,
        compute_advantages,
        ref_model,
        reward_actor,
    ) = await asyncio.gather(
        BlackJackEnvActor.options(**cfg.actors.blackjack_env).as_actor(**cfg.blackjack_env),
        Generator.options(**cfg.services.policy).as_service(**cfg.policy),
        RLTrainer.options(**cfg.actors.trainer).as_actor(**cfg.trainer, loss=simple_grpo_loss),
        ReplayBuffer.options(**cfg.actors.replay_buffer).as_actor(**cfg.replay_buffer, collate=collate),
        ComputeAdvantages.options(**cfg.actors.compute_advantages).as_actor(),
        ReferenceModel.options(**cfg.services.ref_model).as_service(**cfg.ref_model),
        BlackJackReward.options(**cfg.services.reward_actor).as_service(),
    )
    
    print("  ✅ BlackJackEnvActor")
    print("  ✅ Generator (vLLM policy)")
    print("  ✅ RLTrainer")
    print("  ✅ ReplayBuffer")
    print("  ✅ ComputeAdvantages")
    print("  ✅ ReferenceModel")
    print("  ✅ BlackJackReward")
    
    # Initialize torchstore
    trainer_num_procs = cfg.actors.trainer["procs"]
    trainer_host_mesh_name = cfg.actors.trainer["mesh_name"]
    trainer_hosts = provisioner.get_host_mesh(trainer_host_mesh_name)
    await ts.initialize(
        mesh=trainer_hosts.spawn_procs(per_host={"procs": trainer_num_procs}),
        strategy=ts.LocalRankStrategy(),
    )
    print("  ✅ Torchstore")
    
    # Get tokenizer
    tokenizer = await blackjack_env.get_tokenizer.call_one()
    pad_id = await blackjack_env.pad_token.call_one()
    
    print("\n✅ All services initialized!\n")
    
    return {
        'provisioner': provisioner,
        'mlogger': mlogger,
        'blackjack_env': blackjack_env,
        'policy': policy,
        'trainer': trainer,
        'replay_buffer': replay_buffer,
        'compute_advantages': compute_advantages,
        'ref_model': ref_model,
        'reward_actor': reward_actor,
        'tokenizer': tokenizer,
        'pad_id': pad_id,
    }

# Initialize (this will take a minute as models load)
services = await initialize_services(cfg)

## 🔥 Step 5: The Training Loop

<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 20px; border-radius: 10px; color: white; margin: 20px 0;'>
    <h3 style='margin-top: 0;'>⚡ Two Concurrent Loops</h3>
    <ol style='font-size: 16px; line-height: 1.8;'>
        <li><b>Rollouts:</b> Play games → Collect episodes → Add to buffer</li>
        <li><b>Training:</b> Sample batch → Compute loss → Update weights</li>
    </ol>
    <p style='font-size: 14px; margin-top: 15px; opacity: 0.9;'>
        These run in parallel! While games are being played, the trainer is learning from previous games.
    </p>
</div>

Let's define the main training loops using our production utilities.

In [ ]:
async def run_training(cfg, services):
    """Main GRPO training loop."""
    
    # Unpack services
    policy = services['policy']
    trainer = services['trainer']
    replay_buffer = services['replay_buffer']
    compute_advantages = services['compute_advantages']
    ref_model = services['ref_model']
    reward_actor = services['reward_actor']
    tokenizer = services['tokenizer']
    pad_id = services['pad_id']
    mlogger = services['mlogger']
    
    # Training parameters
    group_size = cfg.group_size
    max_req_tokens = cfg.max_req_tokens
    max_res_tokens = cfg.max_res_tokens
    server_url = cfg.blackjack_env.server_url
    max_steps = cfg.trainer.training.steps
    
    shutdown_event = asyncio.Event()
    game_log = setup_game_logger()
    
    # ========================================================================
    # ROLLOUT LOOP: Play games and collect episodes
    # ========================================================================
    async def continuous_rollouts():
        """Collect episodes by playing BlackJack games."""
        rollout_count = 0
        
        while not shutdown_event.is_set():
            t = Tracer("rollouts")
            t.start()
            
            # Play multiple games in parallel
            all_step_results = []
            
            for game_idx in range(group_size):
                game_id = str(uuid.uuid4())[:8]
                
                # Play one game (using production code!)
                step_results = await play_blackjack_game(
                    game_idx=game_idx,
                    game_id=game_id,
                    server_url=server_url,
                    policy=policy,
                    tokenizer=tokenizer,
                    game_log=game_log,
                    rollout_count=rollout_count
                )
                
                all_step_results.extend(step_results)
            
            t.step("play_games")
            
            # Create episodes
            episodes = []
            input_ids = torch.ones(
                (len(all_step_results), max_req_tokens + max_res_tokens),
                dtype=torch.long,
            )
            
            for i, step_result in enumerate(all_step_results):
                episode = Episode(
                    episode_id=str(uuid.uuid4()),
                    pad_id=pad_id,
                    request_len=max_req_tokens,
                    response_len=max_res_tokens,
                    game_id=step_result["game_id"],
                    step_in_game=step_result["step_num"],
                    completion=step_result["response"],
                )
                
                # Evaluate reward
                episode.reward = await reward_actor.evaluate_response.route(
                    prompt=step_result["prompt"],
                    response=step_result["response"].text,
                    game_reward=step_result["final_reward"],
                )
                
                episodes.append(episode)
                input_ids[i, :max_req_tokens] = episode.request_tensor
                input_ids[i, max_req_tokens:] = episode.response_tensor
            
            t.step("reward_eval")
            
            # Get reference logprobs
            ref_logprobs = await ref_model.forward.route(
                input_ids, max_req_tokens, return_logprobs=True
            )
            for i, episode in enumerate(episodes):
                episode.ref_logprobs = ref_logprobs[i]
            
            t.step("ref_model")
            
            # Compute advantages (group-relative!)
            advantages = await compute_advantages.compute.call_one(episodes)
            for episode, advantage in zip(episodes, advantages):
                episode.advantage = advantage
                await replay_buffer.add.call_one(episode)
            
            rollout_count += 1
            t.stop()
            
            # Log summary
            wins = sum(1 for e in episodes if e.reward > 0)
            losses = sum(1 for e in episodes if e.reward < 0)
            print(f"\n📊 Rollout {rollout_count}: {len(episodes)} episodes, "
                  f"W/L: {wins}/{losses}, Win rate: {wins/len(episodes):.1%}")
    
    # ========================================================================
    # TRAINING LOOP: Update policy with GRPO
    # ========================================================================
    async def continuous_training():
        """Train policy on episodes from replay buffer."""
        training_step = 0
        
        while training_step < max_steps:
            t = Tracer("training")
            t.start()
            
            # Sample batch from buffer
            batch = await replay_buffer.sample.call_one(curr_policy_version=training_step)
            if batch is None:
                await asyncio.sleep(0.1)
                continue
            
            t.step("sample_batch")
            
            # Train step
            inputs, targets = batch
            await trainer.train_step.call(inputs, targets)
            training_step += 1
            
            t.step("train")
            
            # Push updated weights
            await trainer.push_weights.call(training_step)
            await policy.update_weights.fanout(training_step)
            
            t.step("update_weights")
            
            # Cleanup old weights
            if training_step >= 2:
                await drop_weights(training_step - 1)
            
            t.stop()
            await mlogger.flush.call_one(training_step)
            
            print(f"✅ Training step {training_step}/{max_steps} complete")
        
        print(f"\n🎉 Training complete! Reached {max_steps} steps.")
    
    # ========================================================================
    # Run both loops concurrently
    # ========================================================================
    print("🚀 Starting GRPO training...\n")
    
    rollout_task = asyncio.create_task(continuous_rollouts())
    training_task = asyncio.create_task(continuous_training())
    
    try:
        await training_task  # Wait for training to finish
    except KeyboardInterrupt:
        print("\n⚠️ Training interrupted")
    finally:
        shutdown_event.set()
        
        # Wait for rollout to finish
        try:
            await asyncio.wait_for(rollout_task, timeout=5)
        except asyncio.TimeoutError:
            rollout_task.cancel()
        
        await shutdown()
        print("\n✅ Shutdown complete")

# Run the training!
await run_training(cfg, services)

## 🎉 Congratulations!

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white; margin: 30px 0;'>
    <h2 style='margin-top: 0;'>You Just Ran Production GRPO! 🚀</h2>
    <p style='font-size: 18px; line-height: 1.8;'>
        This notebook used the <b>exact same code</b> as <code>apps/grpo/blackjack_main_fixed.py</code>.
        Everything you saw is production-ready!
    </p>
</div>

### What You Just Did

✅ Connected to BlackJack via OpenEnv  
✅ Initialized full Forge infrastructure (vLLM, Trainer, ReplayBuffer)  
✅ Ran production GRPO training loop  
✅ Collected episodes with group-relative advantages  
✅ Updated policy with proper KL penalties  

### The Power of This Approach

<table style='width: 100%; border-collapse: collapse; margin: 20px 0;'>
<tr style='background: #667eea; color: white;'>
    <th style='padding: 12px; text-align: left;'>Component</th>
    <th style='padding: 12px; text-align: left;'>What You Get</th>
</tr>
<tr style='background: #f8f9fa;'>
    <td style='padding: 12px;'><b>OpenEnv</b></td>
    <td style='padding: 12px;'>Change one env var → train on 70+ different games</td>
</tr>
<tr style='background: white;'>
    <td style='padding: 12px;'><b>Generator (vLLM)</b></td>
    <td style='padding: 12px;'>Fast inference, multiple replicas, auto weight updates</td>
</tr>
<tr style='background: #f8f9fa;'>
    <td style='padding: 12px;'><b>RLTrainer</b></td>
    <td style='padding: 12px;'>Distributed training with FSDP, gradient accumulation</td>
</tr>
<tr style='background: white;'>
    <td style='padding: 12px;'><b>ReplayBuffer</b></td>
    <td style='padding: 12px;'>Off-policy learning, age-based eviction</td>
</tr>
<tr style='background: #f8f9fa;'>
    <td style='padding: 12px;'><b>Torchstore</b></td>
    <td style='padding: 12px;'>Distributed weight management across replicas</td>
</tr>
</table>

---

## 🚀 Next Steps

### Scale Up the Training

Edit the config to train longer:

```python
cfg.trainer.training.steps = 1000  # More steps
cfg.group_size = 8  # More games per rollout
cfg.rollout_threads = 4  # Parallel rollouts
```

### Try Different Games

Change the OpenEnv game:

```bash
# In terminal:
OPENSPIEL_GAME=tic_tac_toe python -m envs.openspiel_env.server.app
```

Update config:
```python
cfg.blackjack_env.server_url = "http://localhost:8000"
```

**Everything else stays the same!** Same GRPO code works for any game.

### Run the Full Script

For serious training, use the command-line script:

```bash
python -m apps.grpo.blackjack_main_fixed --config apps/grpo/blackjack.yaml
```

This gives you:
- 📊 WandB logging
- 🔥 Multi-GPU support
- 📝 Detailed game logs
- ⚡ Optimized performance

---

## 📚 Resources

- 📦 **OpenEnv**: https://github.com/meta-pytorch/OpenEnv
- 📄 **GRPO Paper**: https://arxiv.org/abs/2402.03300
- 🔧 **Full Script**: `apps/grpo/blackjack_main_fixed.py`
- 🛠️ **Utils**: `apps/grpo/grpo_utils.py`

---

<div style='background: #d4edda; padding: 20px; border-radius: 10px; border-left: 5px solid #28a745; margin: 20px 0;'>
    <h3 style='color: #155724; margin-top: 0;'>🎓 Key Takeaway</h3>
    <p style='color: #155724; font-size: 16px; margin-bottom: 0;'>
        <b>This was REAL production code, not a toy demo.</b><br>
        You just ran the same GRPO implementation that powers large-scale RL training.<br><br>
        The magic? <b>Abstraction.</b> OpenEnv hides game complexity. Forge hides distributed systems complexity.
        You focus on the training loop. 🚀
    </p>
</div>